# 08: REINFORCE（方策勾配法）による倒立振子制御

前回（07）の **Q学習** は「各状態-行動の価値を学ぶ」価値ベースの手法でした。
今回の **REINFORCE** は「方策そのもののパラメータを直接最適化する」方策勾配法です。

| 手法 | アプローチ | 行動空間 | 更新タイミング |
|------|-----------|---------|---------------|
| Q学習（07） | 価値ベース | **離散**（要Discretizer） | ステップごと（TD） |
| **REINFORCE** | 方策勾配 | **連続**（そのまま） | エピソード終了後（MC） |

## 学習内容
1. 方策勾配定理と REINFORCE の理論
2. Q学習との対比（連続行動・オンポリシー）
3. ベースラインによる分散削減
4. 学習曲線と既存ポリシーとの性能比較

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.environments import InvertedPendulumEnv
from src.policies import RandomPolicy, LinearPolicy, REINFORCEPolicy
from src.utils.training import train_reinforce, evaluate_policy

print(f'PyTorch: {torch.__version__}')
print('インポート完了！')

---
## 1. 方策勾配法の理論

### 目的関数

方策パラメータ $\theta$ を使った目的関数 $J(\theta)$ を最大化します：

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} \gamma^t r_t \right]$$

### 方策勾配定理

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

- $G_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$：時刻 $t$ からの割引収益
- $\nabla_\theta \log \pi_\theta(a_t|s_t)$：スコア関数

### REINFORCE の損失関数（PyTorch）

勾配**上昇**（最大化）を勾配**降下**（最小化）に変換：

$$\mathcal{L}(\theta) = -\sum_{t=0}^{T} \log \pi_\theta(a_t|s_t) \cdot G_t$$

### バリアンス削減：ベースライン

$$\mathcal{L}(\theta) = -\sum_{t=0}^{T} \log \pi_\theta(a_t|s_t) \cdot (G_t - b)$$

$b$ はベースライン（今回は $G_t$ の指数移動平均）。
期待値は変わらないが**分散が大きく減る**。

---
## 2. Q学習との対比

In [ ]:
env = InvertedPendulumEnv()
state = env.reset()

print('=== Q学習（07）との対比 ===')
print()
print('【Q学習】行動の離散化が必要だった')
print('  StateDiscretizer → bins=[10,10,20,10] → 20,000状態')
print('  action_values = np.linspace(-10, 10, 11)  ← 11段階に丸める')
print()
print('【REINFORCE】連続行動をそのまま扱える')
print(f'  Gaussian分布 N(μ(s;θ), σ) から直接サンプリング')
print(f'  μ: ネットワーク出力  σ: action_std（ハイパーパラメータ）')
print()

# REINFORCEの行動サンプリングを実演
policy_demo = REINFORCEPolicy(action_std=0.5)
actions = [policy_demo.get_action(state) for _ in range(300)]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(actions, bins=30, edgecolor='white', color='mediumpurple')
ax.set_xlabel('行動（力 N）')
ax.set_ylabel('回数')
ax.set_title('REINFORCE: 初期ネットワークの行動分布（Gaussian、連続値）')
plt.tight_layout()
plt.show()

---
## 3. REINFORCEPolicy の構築

In [ ]:
policy = REINFORCEPolicy(
    hidden_sizes=[64, 64],
    lr=3e-4,
    gamma=0.99,
    action_std=0.5,
    use_baseline=True,      # 移動平均ベースライン有効
    baseline_momentum=0.99,
)

print(policy)
print(f'\nネットワーク構造:')
print(policy._network.get_network_summary())

---
## 4. 学習実行：ベースラインあり vs なし

In [ ]:
# ベースラインあり
policy_with = REINFORCEPolicy(use_baseline=True)
result_with = train_reinforce(
    env, policy_with,
    n_episodes=500, log_interval=100, seed=42
)

print()

# ベースラインなし
policy_without = REINFORCEPolicy(use_baseline=False)
result_without = train_reinforce(
    env, policy_without,
    n_episodes=500, log_interval=100, seed=42
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 報酬の推移
for result, label, color in [
    (result_with,    'ベースラインあり', 'steelblue'),
    (result_without, 'ベースラインなし', 'tomato'),
]:
    rewards = result['episode_rewards']
    ma = result['moving_avg']
    axes[0].plot(rewards, alpha=0.2, color=color)
    axes[0].plot(ma, color=color, lw=2, label=label)

axes[0].set_xlabel('エピソード')
axes[0].set_ylabel('総報酬')
axes[0].set_title('学習曲線（移動平均）')
axes[0].axhline(env.max_steps, ls='--', color='gray', alpha=0.5, label=f'最大({env.max_steps})')
axes[0].legend()

# 損失の推移
for result, label, color in [
    (result_with,    'ベースラインあり', 'steelblue'),
    (result_without, 'ベースラインなし', 'tomato'),
]:
    axes[1].plot(result['episode_losses'], alpha=0.5, color=color, label=label)

axes[1].set_xlabel('エピソード')
axes[1].set_ylabel('Loss')
axes[1].set_title('方策勾配損失の推移')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. 既存ポリシーとの性能比較

In [ ]:
# policy_withを評価（探索なし: action_stdを小さくする）
policy_with.action_std = 0.05

policies_to_compare = {
    'Random':       RandomPolicy(),
    'Linear (手動)': LinearPolicy(weights=np.array([0, 0, 10, 3])),
    'REINFORCE':    policy_with,
}

for name, p in policies_to_compare.items():
    r = evaluate_policy(env, p, n_episodes=50, seed=0)
    print(f'{name:20s}: 平均 {r["mean_reward"]:6.1f} ± {r["std_reward"]:5.1f}')

In [ ]:
comp = {}
policy_with.action_std = 0.05
for name, p in policies_to_compare.items():
    comp[name] = evaluate_policy(env, p, n_episodes=50, seed=0)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c', '#3498db', '#9b59b6']
data   = [comp[k]['episode_rewards'] for k in comp]

bp = ax.boxplot(data, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(list(comp.keys()), fontsize=12)
ax.set_ylabel('総報酬（生存ステップ数）', fontsize=12)
ax.set_title('ポリシー別 性能比較（50エピソード）', fontsize=14)
ax.axhline(env.max_steps, ls='--', color='gray', alpha=0.6, label=f'最大値 ({env.max_steps})')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. まとめと次のステップ

### 3手法の比較まとめ

| 手法 | 更新 | 行動空間 | サンプル効率 | 分散 |
|------|------|---------|------------|------|
| 進化戦略（01-06） | 集団ベース | 連続 | 低 | 低 |
| Q学習（07） | TD（ステップ毎） | 離散化が必要 | 高 | 低 |
| **REINFORCE（08）** | MC（エピソード後） | 連続 | 低 | **高**（要ベースライン） |

### REINFORCEの限界

- **高分散**：エピソード全体の収益を使うため推定がノイジー
- **サンプル非効率**：オンポリシーなため古い経験を再利用できない
- **エピソード終了待ち**：継続タスクに直接適用できない

### 次のステップ：Actor-Critic

- **Critic**（価値ネットワーク）で $G_t$ をステップ毎に推定 → 高分散を解消
- TD更新が可能になり、サンプル効率が大幅改善
- PPO・SAC・DDPGなどの現代的アルゴリズムへの入口